# Training and validation scores at 6 h - bias and variance

Project 19 (air quality forecasting via gradient boosting and LSTM models) - Team T024, Section A.

**Why.** `03_phase1_models.ipynb` and `04_moe.ipynb` report validation and test scores. This notebook puts the
training score of the same models next to them, fold by fold, to read bias and variance: a low training score means
the model cannot fit even the hours it learned from (bias, or noise the inputs cannot explain); a large gap between
training and validation means it fits its training hours too closely (variance).

**What is scored.**

| Model | Recipe |
|---|---|
| persistence | PM2.5 at t + 6 = PM2.5 now |
| blend | Phase 1 model: 0.6 * XGBoost (5 seeds) + 0.4 * Ridge, frozen settings from `reports/phase1/cv_choices.json` |
| moe | final model: quiet expert (blend recipe on hours at or below 150), spike expert (XGBoost on hours above 150), gate, `(1 - p) * quiet + p * spike` |

Training scores are in-sample: each fold's models are refitted on its training hours and scored on those same hours
(only hours where persistence exists, so all three models are scored on the same rows). Validation scores are on the
fold's evaluation year and are checked against the stored ones. For the 2014 models, the fit on 2010 to 2013 is
scored here and their 2014 scores are read from the stored test files.

**Rules.** Nothing here changes a setting or feeds back into the models, and no 2014 hour is scored.

| Step | What |
|---|---|
| 0 | setup, frozen settings |
| 1 | models |
| 2 | training and validation scores per fold |
| 3 | the 2014 models: training fit against the stored test score |
| 4 | reading the numbers |

## 0. Setup

In [1]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from aqf.config import load_config
from aqf.data.pipeline import run
from aqf.data.split import make_folds
from aqf.features import build, fold_xy
from aqf.metrics import score
from aqf.models import baselines, fit_gate, fit_ridge, fit_xgb, gate_blend

pd.set_option("display.width", 120)
pd.set_option("display.precision", 3)

dcfg = load_config("configs/data.yaml")
cfg = load_config("configs/phase1.yaml")
mcfg = load_config("configs/moe.yaml")
OUT = cfg["paths"]["out"]

df = pd.read_parquet(dcfg["paths"]["interim"]) if dcfg["paths"]["interim"].exists() else run(dcfg)[0]
FOLDS = make_folds(df.index, **dcfg["split"])
P1 = json.loads(mcfg["paths"]["phase1"].read_text())
H, EP, xc, SEEDS = mcfg["h"], mcfg["episode"], cfg["xgb"], cfg["ensemble_seeds"]
PARAMS = {**xc["base"], **P1["xgb"][str(H)]}
DELTA = P1["xgb_target"][str(H)] == "change"
ALPHA, RLOG = P1["ridge"][str(H)]["alpha"], P1["ridge"][str(H)]["log_target"]
W = P1["blend"][str(H)]
F, _ = build(df, cfg["features"])
PERSIST = baselines(df, H)["persistence"]
T0 = time.perf_counter()
[f["name"] for f in FOLDS], H, W

(['cv1', 'cv2', 'cv3', 'test'],
 6,
 {'xgb_seeds': 0.6, 'spike': 0.0, 'ridge': 0.4})

## 1. Models

The same recipes as notebooks 03 and 04, refitted per fold. `fit_models` returns one function that predicts with
both the blend and the mixture, so training and validation hours are scored by the same fitted models.

In [2]:
def fit_models(X_tr, y_tr):
    """Blend and mixture fitted on one training span; returns a function that predicts with both."""
    def xgb(rows):
        fits = [fit_xgb(X_tr[rows], y_tr[rows], PARAMS, xc["tail_hours"], H, False, False, s, DELTA)[0] for s in SEEDS]
        return lambda X: np.mean([f.predict(X) for f in fits], axis=0)

    def blend(rows):
        x, r = xgb(rows), fit_ridge(X_tr[rows], y_tr[rows], ALPHA, RLOG)
        return lambda X: W["xgb_seeds"] * x(X) + W["ridge"] * r.predict(X)

    quiet_rows, spike_rows = (y_tr <= EP).to_numpy(), (y_tr > EP).to_numpy()
    full, quiet, spike = blend(np.ones(len(y_tr), bool)), blend(quiet_rows), xgb(spike_rows)
    gates = [fit_gate(X_tr, y_tr, EP, PARAMS, xc["tail_hours"], H, s)[0] for s in SEEDS]

    def predict(X):
        p = np.mean([g.predict_proba(X)[:, 1] for g in gates], axis=0)
        return {"blend": full(X), "moe": gate_blend(quiet(X), spike(X), p, 1.0)}

    return predict


def scores(y, preds, persist):
    ref = score(y, persist)
    rows = [{"model": "persistence", "hours": len(y), **ref, "skill": 0.0}]
    for m, p in preds.items():
        s = score(y, p)
        rows.append({"model": m, "hours": len(y), **s, "skill": 1 - s["rmse"] / ref["rmse"]})
    return rows

## 2. Training and validation scores per fold

Skill is 1 - RMSE / RMSE of persistence on the same hours. The check after the loop pools the validation folds and
compares them with the scores saved by notebooks 03 and 04.

In [3]:
rows = []
for f in FOLDS:
    X_tr, y_tr, X_ev, y_ev = fold_xy(F, df, f, H)
    keep = PERSIST.loc[X_tr.index].notna().to_numpy()
    predict = fit_models(X_tr, y_tr)
    sets = {"train": (X_tr[keep], y_tr[keep])}
    if f["name"] != "test":
        sets["validation"] = (X_ev, y_ev)
    for split, (X, y) in sets.items():
        for r in scores(y, predict(X), PERSIST.loc[X.index]):
            rows.append({"fold": f["name"], "split": split, **r})
    print(f"{f['name']}: done after {time.perf_counter() - T0:.0f} s")

TV = pd.DataFrame(rows)
TV.to_csv(OUT / "train_vs_validation.csv", index=False)

val = TV.query("split == 'validation'")
pooled = {m: np.sqrt((d["hours"] * d["rmse"] ** 2).sum() / d["hours"].sum()) for m, d in val.groupby("model")}
ref_blend = pd.read_csv(OUT / "cv_scores.csv").query("h == @H and model == 'blend'")["rmse"].item()
ref_moe = pd.read_csv(mcfg["paths"]["out"] / "cv_scores.csv").set_index("model").loc["moe_split", "rmse"]
assert np.isclose(pooled["blend"], ref_blend) and np.isclose(pooled["moe"], ref_moe)
print(f"validation RMSE pooled: blend {pooled['blend']:.3f} (stored {ref_blend:.3f}), moe {pooled['moe']:.3f} (stored {ref_moe:.3f})")

cv1: done after 10 s


cv2: done after 23 s


cv3: done after 36 s


test: done after 62 s
validation RMSE pooled: blend 56.330 (stored 56.330), moe 56.256 (stored 56.256)


In [4]:
cv = TV.query("fold != 'test'")
cv.pivot_table(index=["model", "fold"], columns="split", values=["r2", "rmse", "skill"]).loc[["blend", "moe", "persistence"]]

r2               rmse             skill           
split             train validation   train validation  train validation
model       fold                                                       
blend       cv1   0.711      0.639  49.749     56.540  0.301      0.145
            cv2   0.685      0.570  52.151     52.650  0.239      0.136
            cv3   0.685      0.635  50.218     59.405  0.241      0.148
moe         cv1   0.750      0.642  46.287     56.244  0.350      0.150
            cv2   0.719      0.569  49.254     52.660  0.281      0.136
            cv3   0.694      0.634  49.492     59.448  0.252      0.147
persistence cv1   0.408      0.505  71.192     66.148  0.000      0.000
            cv2   0.457      0.423  68.538     60.942  0.000      0.000
            cv3   0.452      0.497  66.175     69.691  0.000      0.000

In [5]:
means = cv.groupby(["model", "split"])[["rmse", "mae", "r2", "skill"]].mean().unstack("split")
means.loc[["blend", "moe", "persistence"]]

rmse                mae                r2             skill           
split         train validation   train validation  train validation  train validation
model                                                                                
blend        50.706     56.198  33.737     36.393  0.694      0.614  0.260      0.143
moe          48.344     56.118  31.414     35.970  0.721      0.615  0.294      0.144
persistence  68.635     65.594  41.620     40.187  0.439      0.475  0.000      0.000

## 3. The 2014 models

Refitted on 2010 to 2013 (the test fold's training span) and scored on those training hours, next to the 2014 scores
stored by notebooks 03 and 04. The 2014 numbers are read from the files, not recomputed.

In [6]:
p1 = pd.read_csv(OUT / "test_scores.csv").query("h == @H").set_index("model")
mt = pd.read_csv(mcfg["paths"]["out"] / "test_scores.csv").set_index("model")
tr = TV.query("fold == 'test'").set_index("model")
pd.DataFrame({
    "R2 train 2010-2013": tr.loc[["blend", "moe"], "r2"].to_numpy(),
    "R2 test 2014 (stored)": [p1.loc["blend", "r2"], mt.loc["moe_split", "r2"]],
    "RMSE train 2010-2013": tr.loc[["blend", "moe"], "rmse"].to_numpy(),
    "RMSE test 2014 (stored)": [p1.loc["blend", "rmse"], mt.loc["moe_split", "rmse"]],
    "skill train": tr.loc[["blend", "moe"], "skill"].to_numpy(),
    "skill test (stored)": [p1.loc["blend", "skill_rmse"], mt.loc["moe_split", "skill_rmse"]],
}, index=["blend", "moe"])

,R2 train 2010-2013,R2 test 2014 (stored),RMSE train 2010-2013,RMSE test 2014 (stored),skill train,skill test (stored)
blend,0.724,0.690,48.195,51.963,0.282,0.151
moe,0.737,0.691,47.092,51.858,0.298,0.153


## 4. Reading the numbers

Means over cv1 to cv3 at 6 h (from the table in step 2):

| Model | R2 train | R2 validation | RMSE train | RMSE validation | Skill train | Skill validation |
|---|--:|--:|--:|--:|--:|--:|
| blend | 0.694 | 0.614 | 50.71 | 56.20 | 26.0% | 14.3% |
| moe | 0.721 | 0.615 | 48.34 | 56.12 | 29.4% | 14.4% |

- **The main limit is information, not overfitting.** Even on the hours it learned from, the blend explains only
  about 69% of the variation. The inputs at hour t cannot say what the weather will do in the next six hours, so most
  likely no model on these inputs can fit that part of the error.
- **Variance is moderate.** Validation R2 is about 0.08 lower and RMSE about 5.5 higher than on training. Part of the
  gap is year-to-year change, not only overfitting, because every validation year is a new year.
- **The 2014 models agree.** Fitted on 2010 to 2013, the blend has training R2 0.724 and test R2 0.690; the mixture
  0.737 and 0.691.
- **The mixture adds variance, not skill.** It fits training hours better than the blend (R2 0.721 against 0.694) but
  is level on validation, so its extra flexibility mostly fits noise, which matches its validation RMSE being within
  seed noise of the blend.

In [7]:
print(f"total {time.perf_counter() - T0:.0f} s")

total 62 s
